# Pre-pipeline sanity checks

Follow-up to `reports/wf_eda_fe_report.md` — validates 4 open questions flagged there
before committing to pipeline/ensemble work: is `relevance_score` alone already
competitive with the embedding baseline, do any papers duplicate across use cases (a
risk for the held-out-use-case design), is the abstract text itself clean, and does the
term-overlap feature just track abstract length. Read-only.

In [1]:
from pathlib import Path
import re
import sys
from collections import defaultdict
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold

DATA_PATH = Path("../../data/processed/papers_combined.parquet")
HOLDOUT_USE_CASE = "tech_forecasting"  # same choice as notebooks/modelling/wf_fold_pca_test.ipynb
N_SPLITS = 5
RANDOM_STATE = 42

plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7",
    "axes.grid": True,
    "grid.color": "#e1e0d9",
    "grid.linewidth": 0.6,
    "axes.axisbelow": True,
    "font.size": 10,
})

df = pd.read_parquet(DATA_PATH)
print(f"Loaded {len(df)} rows x {len(df.columns)} columns")

Loaded 2873 rows x 50 columns


## 1. `relevance_score` alone, under the same fold scheme as the embedding baseline

Reproduces `notebooks/modelling/wf_fold_pca_test.ipynb`'s exact dev-pool/holdout split
and `StratifiedGroupKFold` (same `HOLDOUT_USE_CASE`, `RANDOM_STATE`, grouping/
stratification keys) so the comparison is apples-to-apples: does the existing
`relevance_score` column alone already rival the 0.802 (in-distribution) / 0.530
(holdout) the embeddings scored there?

In [2]:
labelled = df[df["triage_label"].isin(["positive", "negative"])].copy()
labelled["y"] = (labelled["triage_label"] == "positive").astype(int)

dev_pool = labelled[labelled["use_case_key"] != HOLDOUT_USE_CASE].reset_index(drop=True).copy()
holdout = labelled[labelled["use_case_key"] != HOLDOUT_USE_CASE].copy()  # placeholder, fixed below
holdout = labelled[labelled["use_case_key"] == HOLDOUT_USE_CASE].reset_index(drop=True).copy()

def first_author(s):
    if pd.isna(s) or s.strip() == "":
        return None
    return re.sub(r"\s+", " ", s.split(",")[0].strip())

dev_pool["first_author"] = dev_pool["authors"].apply(first_author)
dev_pool["group_key"] = dev_pool["first_author"].fillna(dev_pool["paper_id"])
dev_pool["strat_key"] = dev_pool["use_case_key"] + "__" + dev_pool["triage_label"].astype(str)

X_idx = np.arange(len(dev_pool))
strat_key = dev_pool["strat_key"].to_numpy()
group_key = dev_pool["group_key"].to_numpy()
sgkf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
FOLDS = list(sgkf.split(X_idx, strat_key, group_key))

print(f"Dev pool: {len(dev_pool)} rows, holdout ({HOLDOUT_USE_CASE}): {len(holdout)} rows")
print("(same row counts as wf_fold_pca_test.ipynb confirms this is the identical split)")

Dev pool: 1588 rows, holdout (tech_forecasting): 264 rows
(same row counts as wf_fold_pca_test.ipynb confirms this is the identical split)


In [3]:
# No fitting needed for a precomputed score -- ROC-AUC is invariant to how the score
# was produced, only to its rank order against the label. Direct, no-CV comparison:
dev_pool_auc = roc_auc_score(dev_pool["y"], dev_pool["relevance_score"])
holdout_auc = roc_auc_score(holdout["y"], holdout["relevance_score"])

print("relevance_score alone (raw AUC, no fitting):")
print(f"  dev pool (in-distribution): {dev_pool_auc:.4f}")
print(f"  holdout ({HOLDOUT_USE_CASE}):          {holdout_auc:.4f}")
print()
print("For reference, wf_fold_pca_test.ipynb's embedding-based numbers were:")
print("  grouped CV (in-distribution): 0.8020")
print("  holdout generalisation:       0.5300")
print("  best ensemble (avg of A+B):   0.8340 in-distribution / 0.5860 holdout")

relevance_score alone (raw AUC, no fitting):
  dev pool (in-distribution): 0.6206
  holdout (tech_forecasting):          0.6488

For reference, wf_fold_pca_test.ipynb's embedding-based numbers were:
  grouped CV (in-distribution): 0.8020
  holdout generalisation:       0.5300
  best ensemble (avg of A+B):   0.8340 in-distribution / 0.5860 holdout


In [4]:
# Consistency check: run relevance_score through the EXACT same grouped-CV +
# LogisticRegression machinery as the embedding baseline (single feature). A monotonic
# fit of one feature can't change rank order, so this should reproduce the raw AUC above
# almost exactly -- confirms the fold objects are wired correctly, not a new result.
def cv_roc_auc_grouped_1d(x, y, folds):
    X = x.reshape(-1, 1)
    aucs = []
    for train_idx, test_idx in folds:
        y_test = y[test_idx]
        if len(set(y_test.tolist())) < 2:
            continue
        clf = LogisticRegression(class_weight="balanced", max_iter=1000)
        clf.fit(X[train_idx], y[train_idx])
        pos_col = list(clf.classes_).index(1)
        score = clf.predict_proba(X[test_idx])[:, pos_col]
        aucs.append(roc_auc_score(y_test, score))
    return float(np.mean(aucs))

y_dev = dev_pool["y"].to_numpy()
rel_dev = dev_pool["relevance_score"].to_numpy()
cv_auc_check = cv_roc_auc_grouped_1d(rel_dev, y_dev, FOLDS)
print(f"relevance_score through grouped-CV LogisticRegression: {cv_auc_check:.4f}")
print(f"(vs. raw AUC {dev_pool_auc:.4f} -- should match closely; a big gap would flag a bug)")

relevance_score through grouped-CV LogisticRegression: 0.6212
(vs. raw AUC 0.6206 -- should match closely; a big gap would flag a bug)


## 2. Cross-use-case duplicate check

`explore_use_cases.ipynb`'s fuzzy title match only ever ran **within** one use case, for
DOI-less papers. This checks **across** use cases: the held-out-use-case design in
`wf_fold_pca_test.ipynb` assumes a use case is a genuinely independent slice — a paper
duplicated across two use cases (with usable labels on both sides) would quietly break
that assumption.

In [5]:
dup_doi = df[df["doi"].notna()].groupby("doi")["use_case_key"].nunique()
cross_use_case_dois = dup_doi[dup_doi > 1].index.tolist()
print(f"DOIs appearing under more than one use case: {len(cross_use_case_dois)}")

cross_dup_rows = (
    df[df["doi"].isin(cross_use_case_dois)]
    .sort_values("doi")[["doi", "use_case_key", "triage_label", "title"]]
)
cross_dup_rows

DOIs appearing under more than one use case: 5


,doi,use_case_key,triage_label,title
264,10.1007/s10311-020-01133-3,carbon_capture,positive,Recent advances in carbon capture storage and ...
1367,10.1007/s10311-020-01133-3,cement_binders,NaN,Recent advances in carbon capture storage and ...
2108,10.1007/s40747-025-02137-8,ner,NaN,Temporal knowledge graph completion based on t...
2294,10.1007/s40747-025-02137-8,tech_forecasting,positive,Temporal knowledge graph completion based on t...
2041,10.1145/3696410.3714839,ner,NaN,Tackling Sparse Facts for Temporal Knowledge G...
2339,10.1145/3696410.3714839,tech_forecasting,positive,Tackling Sparse Facts for Temporal Knowledge G...
1946,10.2139/ssrn.5226767,ner,pass,Mpathrp: A Knowledge Graph Relationship Predic...
2232,10.2139/ssrn.5226767,tech_forecasting,pass,Mpathrp: A Knowledge Graph Relationship Predic...
1933,10.3724/jtke-20250008,ner,NaN,Joint entity-relation extraction: A key techni...
2389,10.3724/jtke-20250008,tech_forecasting,positive,Joint entity-relation extraction: A key techni...


In [6]:
# Check whether any of these pairs could actually leak into the current
# wf_fold_pca_test.ipynb result: both sides would need a usable label (positive/negative)
# with one side in dev_pool and the other in the holdout.
usable = {"positive", "negative"}
leak_risk = []
for doi, g in cross_dup_rows.groupby("doi"):
    labels_by_uc = dict(zip(g["use_case_key"], g["triage_label"]))
    usable_sides = {uc: lbl for uc, lbl in labels_by_uc.items() if lbl in usable}
    has_holdout_side = HOLDOUT_USE_CASE in usable_sides
    has_dev_side = any(uc != HOLDOUT_USE_CASE for uc in usable_sides)
    if has_holdout_side and has_dev_side:
        leak_risk.append(doi)

print(f"Of these, {len(leak_risk)} pair(s) have a USABLE (positive/negative) label on "
      f"both a dev-pool use case and the {HOLDOUT_USE_CASE} holdout side -- an active "
      f"leakage risk in the current split.")
print("(0 expected here: inspect the table above -- the non-holdout side of every pair "
      "is NaN or 'pass', already excluded from the labelled training subset, so this "
      "batch of duplicates happens not to be active right now. A structural risk for "
      "future re-labelling, not a live bug today.)" if len(leak_risk) == 0 else "ACTIVE LEAKAGE -- see doi list above.")

Of these, 0 pair(s) have a USABLE (positive/negative) label on both a dev-pool use case and the tech_forecasting holdout side -- an active leakage risk in the current split.
(0 expected here: inspect the table above -- the non-holdout side of every pair is NaN or 'pass', already excluded from the labelled training subset, so this batch of duplicates happens not to be active right now. A structural risk for future re-labelling, not a live bug today.)


In [7]:
# Fuzzy near-duplicate check across use cases, for papers WITHOUT a shared DOI.
# Blocked by year (a genuine duplicate should report the same year in both exports) and
# by a shared-significant-word prefilter (cuts ~1.4M possible pairs down to a tractable
# candidate set before the expensive SequenceMatcher call) -- same normalize_title /
# threshold=0.85 convention as explore_use_cases.ipynb, just applied across use cases.
FUZZY_MATCH_THRESHOLD = 0.85

def normalize_title(title):
    title = (title or "").lower().strip()
    title = re.sub(r"[^\w\s]", "", title)
    title = re.sub(r"\s+", " ", title)
    return title

def significant_words(title_norm, min_len=5):
    return {w for w in title_norm.split() if len(w) >= min_len}

work = df[["paper_id", "use_case_key", "year", "doi", "title"]].copy()
work["title_norm"] = work["title"].map(normalize_title)
work["sig_words"] = work["title_norm"].map(significant_words)

already_found_dois = set(cross_use_case_dois)
candidates = []
for year, g in work.dropna(subset=["year"]).groupby("year"):
    g = g.reset_index(drop=True)
    word_index = defaultdict(list)
    for i, words in enumerate(g["sig_words"]):
        for w in words:
            word_index[w].append(i)
    pair_shared_count = defaultdict(int)
    for idxs in word_index.values():
        for a in range(len(idxs)):
            for b in range(a + 1, len(idxs)):
                pair_shared_count[(idxs[a], idxs[b])] += 1
    for (i, j), shared in pair_shared_count.items():
        if shared < 2:
            continue
        if g.loc[i, "use_case_key"] == g.loc[j, "use_case_key"]:
            continue
        doi_i, doi_j = g.loc[i, "doi"], g.loc[j, "doi"]
        if pd.notna(doi_i) and doi_i == doi_j:
            continue  # already caught by the exact-DOI check above -- not a new finding
        score = SequenceMatcher(None, g.loc[i, "title_norm"], g.loc[j, "title_norm"]).ratio()
        if score >= FUZZY_MATCH_THRESHOLD:
            candidates.append({
                "year": year,
                "use_case_1": g.loc[i, "use_case_key"], "paper_id_1": g.loc[i, "paper_id"], "title_1": g.loc[i, "title"],
                "use_case_2": g.loc[j, "use_case_key"], "paper_id_2": g.loc[j, "paper_id"], "title_2": g.loc[j, "title"],
                "similarity": round(score, 3),
            })

fuzzy_cross_uc = pd.DataFrame(candidates).sort_values("similarity", ascending=False) if candidates else pd.DataFrame()
print(f"Cross-use-case fuzzy title matches, DIFFERENT dois or DOI-less (threshold={FUZZY_MATCH_THRESHOLD}), "
      f"excluding the {len(already_found_dois)} exact-DOI pairs already found above: {len(fuzzy_cross_uc)}")
fuzzy_cross_uc

Cross-use-case fuzzy title matches, DIFFERENT dois or DOI-less (threshold=0.85), excluding the 5 exact-DOI pairs already found above: 0


""


## 3. Abstract text quality

Never explicitly audited: does the raw `abstract` text carry scraped-in contamination
(publisher boilerplate, HTML/LaTeX remnants) or look truncated? This feeds the
embeddings and the term-overlap feature directly.

In [8]:
has_abs = df["abstract"].fillna("").str.strip() != ""
abstracts = df.loc[has_abs, ["use_case_key", "abstract"]].copy()
print(f"{len(abstracts)} / {len(df)} rows have a non-empty abstract to audit")

PATTERNS = {
    "copyright_boilerplate": r"©|all rights reserved|elsevier|springer nature|licensee mdpi|creative commons|copyright \d{4}|published by",
    "html_tags": r"<\/?[a-zA-Z][^>]{0,30}>",
    "latex_artifacts": r"\$[^$]{1,40}\$|\\[a-zA-Z]+\{",
}

for name, pattern in PATTERNS.items():
    hit = abstracts["abstract"].str.contains(pattern, case=False, regex=True, na=False)
    abstracts[name] = hit

contamination_by_uc = abstracts.groupby("use_case_key")[list(PATTERNS)].mean().round(4)
print("Contamination rate by use case (share of abstracts matching each pattern):")
contamination_by_uc

2302 / 2873 rows have a non-empty abstract to audit
Contamination rate by use case (share of abstracts matching each pattern):


,copyright_boilerplate,html_tags,latex_artifacts
use_case_key,,,
carbon_capture,0.0074,0.0,0.0222
cement_binders,0.0016,0.0,0.0164
ner,0.0041,0.0,0.0289
soil_microbiome,0.0000,0.0,0.0000
solar_leo,0.0085,0.0,0.0114
tech_forecasting,0.0041,0.0,0.0163


In [9]:
abstracts["word_count"] = abstracts["abstract"].str.split().str.len()
abstracts["ends_mid_sentence"] = ~abstracts["abstract"].str.strip().str.endswith((".", "!", "?", '"', "\u201d"))

print("Abstract word-count distribution:")
print(abstracts["word_count"].describe().round(1))
print()
short_rate = (abstracts["word_count"] < 20).mean()
print(f"Abstracts under 20 words (possible truncation / near-empty): {short_rate:.1%}")
print(f"Abstracts not ending in terminal punctuation (soft truncation signal): "
      f"{abstracts['ends_mid_sentence'].mean():.1%}")
print()
print("NOTE: 'not ending in terminal punctuation' is a soft signal, not proof of "
      "truncation -- some abstracts are legitimately formatted without one. Read the "
      "examples below before treating this as a hard defect count.")

Abstract word-count distribution:
count    2302.0
mean      197.7
std        77.3
min         1.0
25%       154.0
50%       193.0
75%       233.0
max      1085.0
Name: word_count, dtype: float64

Abstracts under 20 words (possible truncation / near-empty): 0.9%
Abstracts not ending in terminal punctuation (soft truncation signal): 4.8%

NOTE: 'not ending in terminal punctuation' is a soft signal, not proof of truncation -- some abstracts are legitimately formatted without one. Read the examples below before treating this as a hard defect count.


In [10]:
any_contam = abstracts[list(PATTERNS)].any(axis=1)
print(f"Any contamination pattern matched: {any_contam.sum()} / {len(abstracts)} ({any_contam.mean():.1%})")
print()
print("Examples:")
for _, row in abstracts[any_contam].head(5).iterrows():
    matched = [k for k in PATTERNS if row[k]]
    preview = row["abstract"][:220].replace("\n", " ")
    print(f"[{row['use_case_key']}] matched={matched}\n  {preview}...\n")

Any contamination pattern matched: 47 / 2302 (2.0%)

Examples:
[carbon_capture] matched=['latex_artifacts']
  Carbon capture via chemical absorption is critical for carbon neutrality but faces deployment barriers including high-energy consumption, high cost, and insufficient system integration. This review establishes a three-di...

[carbon_capture] matched=['copyright_boilerplate']
  Abstract Carbon capture technologies have been recognized as a potential alternative to alleviate global warming. Carbon capture and storage (CCS) is preferred over carbon conversion and utilization (CCU) due to its lowe...

[carbon_capture] matched=['latex_artifacts']
  This study focuses on membrane-based systems for CO$_2$ separation, addressing the urgent need for efficient carbon capture solutions to mitigate climate change. Linear regression models, based on membrane equations, wer...

[carbon_capture] matched=['latex_artifacts']
  Intensive energy penalty caused by CO2 separation process is a criti

## 4. Does term-overlap just track abstract length?

Reuses `terms_overlap.ipynb`'s exact `overlap_score` construction (whole-word matching,
`must_fraction + 0.5*nice_fraction - exclude_hit`) on the 3 use cases where it found
real signal (cement_binders, carbon_capture, ner). A longer abstract has a mechanically
higher chance of containing a must-include term regardless of actual topical fit --
checking whether that's what's actually driving the reported AUCs.

In [11]:
def normalize(text):
    return text.lower() if isinstance(text, str) else ""

def whole_word_present(term, text_norm):
    if not term:
        return False
    pattern = r"(?<!\w)" + re.escape(term.lower()) + r"(?!\w)"
    return re.search(pattern, text_norm) is not None

def term_fraction(terms, text_norm):
    terms = list(terms) if terms is not None else []
    if len(terms) == 0:
        return np.nan
    return sum(whole_word_present(t, text_norm) for t in terms) / len(terms)

def any_present(terms, text_norm):
    terms = list(terms) if terms is not None else []
    if len(terms) == 0:
        return np.nan
    return float(any(whole_word_present(t, text_norm) for t in terms))

WORKING_USE_CASES = ["cement_binders", "carbon_capture", "ner"]
check_df = df[df["use_case_key"].isin(WORKING_USE_CASES)].copy()
check_df["text_norm"] = (check_df["title"].fillna("") + ". " + check_df["abstract"].fillna("")).map(normalize)
check_df["word_count"] = check_df["text_norm"].str.split().str.len()

check_df["must_fraction"] = [term_fraction(t, tx) for t, tx in zip(check_df["terms_must_include"], check_df["text_norm"])]
check_df["nice_fraction"] = [term_fraction(t, tx) for t, tx in zip(check_df["terms_nice_to_have"], check_df["text_norm"])]
check_df["exclude_hit"] = [any_present(t, tx) for t, tx in zip(check_df["terms_exclude"], check_df["text_norm"])]
check_df["overlap_score"] = (
    check_df["must_fraction"].fillna(0) + 0.5 * check_df["nice_fraction"].fillna(0) - check_df["exclude_hit"].fillna(0)
)

print("Correlation between overlap_score and word_count, per use case:")
for uc, g in check_df.groupby("use_case_key"):
    r, p = stats.pearsonr(g["word_count"], g["overlap_score"])
    rho, ps = stats.spearmanr(g["word_count"], g["overlap_score"])
    print(f"  {uc:16s} pearson r={r:+.3f} (p={p:.3g})   spearman rho={rho:+.3f} (p={ps:.3g})")

Correlation between overlap_score and word_count, per use case:
  carbon_capture   pearson r=+0.344 (p=2.02e-10)   spearman rho=+0.369 (p=7.39e-12)
  cement_binders   pearson r=-0.039 (p=0.305)   spearman rho=+0.017 (p=0.654)
  ner              pearson r=+0.082 (p=0.0563)   spearman rho=+0.080 (p=0.0644)


In [12]:
# The sharper question: does word_count ALONE already predict triage_label?
# If it does at anywhere near overlap_score's reported AUC, overlap_score risks being
# a word-count proxy dressed up as a topical-fit score.
labeled_check = check_df[check_df["triage_label"].isin(["positive", "negative"])].copy()
y_check = (labeled_check["triage_label"] == "positive").astype(int)

print("ROC-AUC of word_count alone vs. overlap_score alone, per use case:")
for uc, g in labeled_check.groupby("use_case_key"):
    y_uc = (g["triage_label"] == "positive").astype(int)
    auc_wc = roc_auc_score(y_uc, g["word_count"])
    auc_ov = roc_auc_score(y_uc, g["overlap_score"])
    print(f"  {uc:16s} word_count AUC={auc_wc:.3f}   overlap_score AUC={auc_ov:.3f}   (reported in terms_overlap.ipynb: 0.80/0.71/0.70)")

ROC-AUC of word_count alone vs. overlap_score alone, per use case:
  carbon_capture   word_count AUC=0.528   overlap_score AUC=0.709   (reported in terms_overlap.ipynb: 0.80/0.71/0.70)
  cement_binders   word_count AUC=0.472   overlap_score AUC=0.801   (reported in terms_overlap.ipynb: 0.80/0.71/0.70)
  ner              word_count AUC=0.509   overlap_score AUC=0.698   (reported in terms_overlap.ipynb: 0.80/0.71/0.70)


## Summary

| Check | Finding | Verdict |
|---|---|---|
| `relevance_score` alone vs. embedding baseline | See §1 printed numbers | See notebook output |
| Cross-use-case duplicates | 5 exact-DOI collisions found (mostly ner ↔ tech_forecasting); 0 create active leakage in the current split | Structural risk flagged, not a live bug |
| Abstract text quality | See §3 contamination rates and examples | See notebook output |
| Term-overlap vs. abstract length | See §4 correlation and word-count-alone AUC | See notebook output |

Numbers deliberately not restated here — read the printed output above, it's the
primary source, not this table.